# CIC DataSense IIoT 2025: Adaptive Fisher-Weighted Z Simulation with Rolling Statistics

This notebook reproduces the final CIC DataSense IIoT 2025 experiment used to evaluate the adaptive client-specific anomaly-detection framework.

### Experiment flow
1. **Load and split the data** into disjoint training, tuning, enrollment, benign-test, and attack-test partitions.
2. **Build general device-type profiles** from known benign devices and learn fixed Fisher feature weights.
3. **Create held-out client profiles** that start from the corresponding general device-type profile and adapt online using compact rolling statistics.
4. **Tune seven adaptation parameters with PSO** using only the tuning partitions; final benign and attack test data are not used during optimization.
5. **Calibrate each held-out client** from trusted enrollment data and the benign tuning stream.
6. **Evaluate three conditions**: static baseline, static model under client/device drift, and the adaptive client-specific model. Attack detection is also evaluated on the untouched attack-test split.
7. **Apply the 2-of-3 evidence rule**, calculate FPR/detection/runtime/storage metrics, and save the final result CSVs.

### Reproducibility notes
- The notebook expects `benign_samples_10sec.csv`, `attack_samples_10sec.csv`, and `profiles.py` in the same working directory.
- Fisher weights are learned offline and remain fixed during online evaluation.
- Online adaptation updates only client means, standard deviations, and thresholds; it does **not** retrain a classifier.
- Randomized attack splitting and PSO both use fixed seeds for repeatability.


## 1. Load data and create disjoint experiment splits

This section defines the 25 network-behavior features, maps concrete devices to broader device types, and separates devices into two roles:

- **Profile devices** are known devices used to construct general device-type behavior profiles.
- **Held-out/drift devices** represent previously unseen clients of those device types and are used for enrollment, tuning, and final streaming evaluation.

Benign rows are split **chronologically within each device** so later observations never leak into earlier training stages. Attack rows are split within each `(device, attack family, attack type)` group so non-singleton attack groups are represented in both the tuning and final-test partitions.


In [1]:
from pathlib import Path
import copy
import time

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

from profiles import DeviceTypes, ClientProfiles


# Input/output paths. Run the notebook from this directory so these relative paths resolve.
DATA_DIR = Path(".")
BENIGN_FILE = DATA_DIR / "benign_samples_10sec.csv"
ATTACK_FILE = DATA_DIR / "attack_samples_10sec.csv"
RESULTS_DIR = DATA_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# Network-behavior statistics extracted from each 10-second traffic window.
# The same ordered feature list is used by every profile and scoring function.
FEATURE_COLUMNS = [
    "network_packets_all_count",
    "network_packets_src_count",
    "network_packets_dst_count",
    "network_interval-packets",
    "network_time-delta_avg",
    "network_packet-size_avg",
    "network_payload-length_avg",
    "network_ip-length_avg",
    "network_header-length_avg",
    "network_ips_all_count",
    "network_ips_src_count",
    "network_ips_dst_count",
    "network_ports_all_count",
    "network_ports_src_count",
    "network_ports_dst_count",
    "network_protocols_all_count",
    "network_tcp-flags-ack_count",
    "network_tcp-flags-syn_count",
    "network_tcp-flags-fin_count",
    "network_tcp-flags-rst_count",
    "network_tcp-flags-psh_count",
    "network_ttl_avg",
    "network_window-size_avg",
    "network_fragmented-packets",
    "network_fragmentation-score",
]

# Map concrete CIC DataSense device names to the broader device types modeled
# by the framework. General profiles are learned at this device-type level.
DEVICE_TYPE_MAP = {
    "accelerometer-sensor": "sensor",
    "flame-sensor": "sensor",
    "gas-sensor": "sensor",
    "light-sensor": "sensor",
    "motion-sensor": "sensor",
    "proximity-collision-sensor": "sensor",
    "rfid-sensor": "sensor",
    "soil-sensor": "sensor",
    "sound-sensor": "sensor",
    "steam-sensor": "sensor",
    "ultrasonic-sensor": "sensor",
    "vibration-sensor": "sensor",
    "water-sensor": "sensor",
    "weather-sensor": "sensor",
    "blurams-camera": "camera",
    "dekco-camera": "camera",
    "geeni-camera": "camera",
    "myq-camera": "camera",
    "wisenet-camera": "camera",
    "yi-camera": "camera",
    "plug-all-cameras": "smart-plug",
    "plug-all-rpb": "smart-plug",
    "plug-all-sensors": "smart-plug",
    "plug-cameras-dekco-blurams": "smart-plug",
    "plug-cameras-geeni": "smart-plug",
    "plug-cameras-yi": "smart-plug",
    "plug-edge1": "smart-plug",
    "plug-flame": "smart-plug",
    "plug-motion": "smart-plug",
    "plug-mqtt": "smart-plug",
    "plug-proximity": "smart-plug",
    "plug-rfid": "smart-plug",
    "plug-vibration": "smart-plug",
    "router": "network-device",
    "switch": "network-device",
    "ap": "network-device",
    "mqtt-broker": "edge-infrastructure",
    "edge1": "edge-infrastructure",
}

# Known devices used to learn the initial/general device-type profiles.
PROFILE_DEVICES = [
    "accelerometer-sensor",
    "flame-sensor",
    "gas-sensor",
    "light-sensor",
    "motion-sensor",
    "proximity-collision-sensor",
    "rfid-sensor",
    "blurams-camera",
    "dekco-camera",
    "geeni-camera",
    "plug-all-cameras",
    "plug-all-rpb",
    "plug-all-sensors",
    "plug-cameras-dekco-blurams",
    "plug-cameras-geeni",
    "plug-cameras-yi",
    "router",
    "mqtt-broker",
]
# All remaining devices are held out from general-profile training. They act
# as unseen clients whose behavior can differ from the device-type baseline.
DRIFT_DEVICES = sorted(set(DEVICE_TYPE_MAP) - set(PROFILE_DEVICES))


def chronological_parts(dataframe, fractions, names):
    """Split each device chronologically according to the requested fractions.

    Splitting independently within each device preserves temporal ordering and
    prevents future observations from leaking into earlier experiment stages.
    """
    parts = {name: [] for name in names}
    for _, rows in dataframe.groupby("device_name", sort=False):
        rows = rows.sort_values("timestamp_start", kind="stable").copy()
        cuts = np.floor(np.cumsum(fractions[:-1]) * len(rows)).astype(int)
        boundaries = [0, *cuts.tolist(), len(rows)]
        for name, start, stop in zip(names, boundaries[:-1], boundaries[1:]):
            parts[name].append(rows.iloc[start:stop].copy())
    return {
        name: pd.concat(items, ignore_index=True)
        for name, items in parts.items()
    }


def grouped_attack_split(
    dataframe,
    group_columns,
    tuning_fraction=0.20,
    seed=42,
):
    """Keep every non-singleton attack group in both tuning and final test."""
    rng = np.random.default_rng(seed)
    tuning_indices = []
    test_indices = []

    for _, rows in dataframe.groupby(group_columns, sort=False):
        indices = rows.index.to_numpy()
        shuffled = rng.permutation(indices)
        if len(shuffled) == 1:
            test_indices.extend(shuffled)
            continue

        tuning_count = int(round(tuning_fraction * len(shuffled)))
        tuning_count = min(len(shuffled) - 1, max(1, tuning_count))
        tuning_indices.extend(shuffled[:tuning_count])
        test_indices.extend(shuffled[tuning_count:])

    tuning = dataframe.loc[tuning_indices].sort_index(kind="stable").copy()
    test = dataframe.loc[test_indices].sort_index(kind="stable").copy()
    return tuning, test


# ------------------------------ Benign data ------------------------------
# Read only the columns required by the experiment, coerce feature values to
# numeric form, then drop malformed/incomplete windows before splitting.
benign_columns = [
    "device_name",
    "timestamp_start",
    *FEATURE_COLUMNS,
]
benign_df = pd.read_csv(BENIGN_FILE, usecols=benign_columns)
benign_df["device_type"] = benign_df["device_name"].map(DEVICE_TYPE_MAP)
benign_df["timestamp_start"] = pd.to_datetime(
    benign_df["timestamp_start"],
    utc=True,
    errors="coerce",
)
benign_df[FEATURE_COLUMNS] = benign_df[FEATURE_COLUMNS].apply(
    pd.to_numeric,
    errors="coerce",
)
benign_df = benign_df.dropna(
    subset=["device_type", "timestamp_start", *FEATURE_COLUMNS]
).sort_values(["device_name", "timestamp_start"], kind="stable")
benign_df["position"] = benign_df.groupby("device_name").cumcount()

known_df = benign_df[benign_df["device_name"].isin(PROFILE_DEVICES)].copy()
heldout_df = benign_df[benign_df["device_name"].isin(DRIFT_DEVICES)].copy()

# Score configuration receives its own known-device validation partition.
# The last 20% remains an untouched baseline test, as in the original notebook.
known_parts = chronological_parts(
    known_df,
    fractions=[0.70, 0.10, 0.20],
    names=["general_train", "scoring_validation", "baseline_test"],
)
heldout_parts = chronological_parts(
    heldout_df,
    fractions=[0.20, 0.20, 0.60],
    names=["enrollment", "tuning", "stream"],
)
general_train_df = known_parts["general_train"]
scoring_validation_df = known_parts["scoring_validation"]
baseline_test_df = known_parts["baseline_test"]
enrollment_df = heldout_parts["enrollment"]
tuning_df = heldout_parts["tuning"]
stream_df = heldout_parts["stream"]

# ------------------------------ Attack data ------------------------------
# Attack data is used in two disjoint roles: a tuning subset for Fisher/PSO
# configuration and an untouched test subset for final detection results.
attack_columns = [
    "device_name",
    "label2",
    "label3",
    *FEATURE_COLUMNS,
]
# The source file ends with one incomplete quoted row; the Python parser
# safely skips that malformed final record.
attack_df = pd.read_csv(
    ATTACK_FILE,
    usecols=attack_columns,
    engine="python",
    on_bad_lines="skip",
)
attack_df = attack_df.rename(columns={
    "label2": "attack_family",
    "label3": "attack_type",
})
attack_df["device_type"] = attack_df["device_name"].map(DEVICE_TYPE_MAP)
attack_df[FEATURE_COLUMNS] = attack_df[FEATURE_COLUMNS].apply(
    pd.to_numeric,
    errors="coerce",
)
attack_df = attack_df[
    attack_df["device_name"].isin(DRIFT_DEVICES)
].dropna(
    subset=[
        "device_type",
        "attack_family",
        "attack_type",
        *FEATURE_COLUMNS,
    ]
).copy()
attack_df["attack_position"] = attack_df.groupby("device_name").cumcount()

# Split inside each exact group. This prevents common attacks from dominating
# tuning and leaves every non-singleton group represented in the final test.
attack_tuning_df, attack_test_df = grouped_attack_split(
    attack_df,
    group_columns=[
        "device_name",
        "attack_family",
        "attack_type",
    ],
    tuning_fraction=0.20,
    seed=42,
)

assert set(attack_tuning_df.index).isdisjoint(attack_test_df.index)

split_summary = pd.DataFrame([
    {"split": "general profile training", "rows": len(general_train_df)},
    {"split": "score-configuration validation", "rows": len(scoring_validation_df)},
    {"split": "known-device baseline test", "rows": len(baseline_test_df)},
    {"split": "held-out enrollment", "rows": len(enrollment_df)},
    {"split": "held-out PSO/client tuning", "rows": len(tuning_df)},
    {"split": "held-out final benign stream", "rows": len(stream_df)},
    {"split": "attack tuning", "rows": len(attack_tuning_df)},
    {"split": "untouched attack test", "rows": len(attack_test_df)},
])
display(split_summary)


,split,rows
0,general profile training,4518
1,score-configuration validation,666
2,known-device baseline test,1296
3,held-out enrollment,1440
4,held-out PSO/client tuning,1440
5,held-out final benign stream,4320
6,attack tuning,1598
7,untouched attack test,6858


## 2. Build Fisher-weighted general device-type profiles

For each general device type, this section learns:

- the benign per-feature mean and standard deviation;
- a fixed Fisher weight for each feature, measuring how strongly it separates benign traffic from the attack-tuning comparison data; and
- a 99th-percentile anomaly threshold from the benign training scores.

Type-specific attack-tuning data is used for Fisher weighting when available. If a device type has no matching attack-tuning rows, the pooled attack-tuning partition is used as a fallback. The untouched final attack-test split is never used here.


In [2]:
# Fisher weights emphasize features that consistently separate benign and
# attack traffic. They are learned once from the disjoint attack-tuning split
# and remain fixed while each client's statistics and threshold adapt online.
SCORING_PERCENTILE = 99.0
SCORE_STD_FLOOR = 0.1
SCORING_FPR_TARGET = 0.01


def _safe_feature_stds(values, std_floor):
    """Return finite per-feature standard deviations with a minimum floor."""
    values = np.asarray(values, dtype=float)
    ddof = 1 if len(values) > 1 else 0
    stds = values.std(axis=0, ddof=ddof)
    stds = np.nan_to_num(
        stds,
        nan=std_floor,
        posinf=std_floor,
        neginf=std_floor,
    )
    return np.maximum(stds, std_floor)


def fisher_weights(benign_values, comparison_values):
    """Learn normalized Fisher separation weights without iterative training."""
    benign_values = np.asarray(benign_values, dtype=float)
    comparison_values = np.asarray(comparison_values, dtype=float)

    benign_means = benign_values.mean(axis=0)
    benign_stds = _safe_feature_stds(benign_values, SCORE_STD_FLOOR)
    comparison_means = comparison_values.mean(axis=0)
    comparison_stds = _safe_feature_stds(
        comparison_values,
        SCORE_STD_FLOOR,
    )

    pooled = np.sqrt(benign_stds**2 + comparison_stds**2)
    weights = np.abs(comparison_means - benign_means) / np.maximum(
        pooled,
        SCORE_STD_FLOOR,
    )
    weights = np.maximum(weights, np.finfo(float).eps)
    return weights / weights.mean()


def fisher_weighted_scores(values, means, stds, weights):
    """Return the Fisher-weighted mean absolute Z score."""
    values = np.asarray(values, dtype=float)
    one_row = values.ndim == 1
    values = np.atleast_2d(values)
    z_scores = np.abs(values - means) / np.maximum(
        stds,
        SCORE_STD_FLOOR,
    )
    scores = np.average(z_scores, axis=1, weights=weights)
    return float(scores[0]) if one_row else scores


def profile_arrays(profile):
    """Return profile means/stds as arrays aligned to FEATURE_COLUMNS."""
    means = np.array(
        [profile.feature_means[name] for name in FEATURE_COLUMNS],
        dtype=float,
    )
    stds = np.array(
        [profile.feature_stds[name] for name in FEATURE_COLUMNS],
        dtype=float,
    )
    return means, np.maximum(stds, SCORE_STD_FLOOR)


def profile_weights(profile):
    """Return profile Fisher weights as an array aligned to FEATURE_COLUMNS."""
    return np.array(
        [profile.feature_weights[name] for name in FEATURE_COLUMNS],
        dtype=float,
    )


def scoring_detection_metrics(flags, rows):
    """Protect clients and attack types separately, not tiny intersections."""
    scored = rows[["device_name", "attack_type"]].copy()
    scored["flagged"] = np.asarray(flags, dtype=bool)
    device_rates = scored.groupby(
        "device_name",
        sort=False,
    )["flagged"].mean()
    attack_type_rates = scored.groupby(
        "attack_type",
        sort=False,
    )["flagged"].mean()
    all_group_rates = pd.concat(
        [device_rates, attack_type_rates],
        ignore_index=True,
    )
    return {
        "attack_micro_detection": float(scored["flagged"].mean()),
        "attack_macro_detection": float(all_group_rates.mean()),
        "worst_group_detection": float(all_group_rates.min()),
        "worst_client_detection": float(device_rates.min()),
        "worst_attack_type_detection": float(attack_type_rates.min()),
    }


# Build one fixed general profile per device type. These objects provide the
# initialization/anchor for every held-out client of that type.
general_profiles = {}
FISHER_THRESHOLD_BY_TYPE = {}
profile_rows = []
weight_rows = []
pooled_attack_values = attack_tuning_df[
    FEATURE_COLUMNS
].to_numpy(dtype=float)

for device_type, train_rows in general_train_df.groupby(
    "device_type",
    sort=False,
):
    train_values = train_rows[FEATURE_COLUMNS].to_numpy(dtype=float)
    validation_rows = scoring_validation_df[
        scoring_validation_df["device_type"] == device_type
    ]
    validation_values = validation_rows[
        FEATURE_COLUMNS
    ].to_numpy(dtype=float)
    attack_rows = attack_tuning_df[
        attack_tuning_df["device_type"] == device_type
    ]

    # Prefer attack examples from the same device type when they exist.
    # Pooled tuning attacks are used only when no type-specific rows exist.
    if len(attack_rows):
        comparison_values = attack_rows[
            FEATURE_COLUMNS
        ].to_numpy(dtype=float)
        weight_source = "device-type attack tuning"
    elif len(pooled_attack_values):
        comparison_values = pooled_attack_values
        weight_source = "pooled attack tuning fallback"
    else:
        raise ValueError(
            "Adaptive Fisher-weighted Z requires a non-empty attack-tuning split."
        )

    means = train_values.mean(axis=0)
    stds = _safe_feature_stds(train_values, SCORE_STD_FLOOR)
    weights = fisher_weights(train_values, comparison_values)

    train_scores = fisher_weighted_scores(
        train_values,
        means,
        stds,
        weights,
    )
    validation_scores = fisher_weighted_scores(
        validation_values,
        means,
        stds,
        weights,
    )
    # The general threshold is calibrated only from benign training scores.
    threshold = float(
        np.percentile(train_scores, SCORING_PERCENTILE)
    )
    validation_fpr = float(
        np.mean(validation_scores > threshold)
    )

    if len(attack_rows):
        attack_scores = fisher_weighted_scores(
            comparison_values,
            means,
            stds,
            weights,
        )
        detection = scoring_detection_metrics(
            attack_scores > threshold,
            attack_rows,
        )
    else:
        detection = {
            "attack_micro_detection": np.nan,
            "attack_macro_detection": np.nan,
            "worst_group_detection": np.nan,
            "worst_client_detection": np.nan,
            "worst_attack_type_detection": np.nan,
        }

    profile = DeviceTypes(
        device_type_name=device_type,
        feature_means=dict(zip(FEATURE_COLUMNS, means)),
        feature_stds=dict(zip(FEATURE_COLUMNS, stds)),
        feature_weights=dict(zip(FEATURE_COLUMNS, weights)),
        threshold=threshold,
        std_floor=SCORE_STD_FLOOR,
    )
    general_profiles[device_type] = profile
    FISHER_THRESHOLD_BY_TYPE[device_type] = threshold

    profile_rows.append({
        "device_type": device_type,
        "scoring_method": "Fisher-Weighted Z",
        "weight_source": weight_source,
        "training_rows": len(train_rows),
        "threshold": threshold,
        "validation_fpr": validation_fpr,
        **detection,
    })

    for feature, weight in zip(FEATURE_COLUMNS, weights):
        weight_rows.append({
            "device_type": device_type,
            "feature": feature,
            "fisher_weight": float(weight),
            "weight_source": weight_source,
        })

general_profile_summary = pd.DataFrame(profile_rows)
fisher_feature_weights = pd.DataFrame(weight_rows)
display(general_profile_summary)
display(
    fisher_feature_weights.sort_values(
        ["device_type", "fisher_weight"],
        ascending=[True, False],
        kind="stable",
    ).groupby("device_type", sort=False).head(5)
)


,device_type,scoring_method,weight_source,training_rows,threshold,validation_fpr,attack_micro_detection,attack_macro_detection,worst_group_detection,worst_client_detection,worst_attack_type_detection
0,sensor,Fisher-Weighted Z,device-type attack tuning,1757,1.371074,0.003861,0.881877,0.866417,0.0,0.838384,0.0
1,camera,Fisher-Weighted Z,device-type attack tuning,753,1.518058,0.000000,0.903915,0.893462,0.0,0.893204,0.0
2,edge-infrastructure,Fisher-Weighted Z,device-type attack tuning,251,3.904404,0.000000,1.000000,1.000000,1.0,1.000000,1.0
3,smart-plug,Fisher-Weighted Z,device-type attack tuning,1506,1.895912,0.022523,0.333333,0.533681,0.0,0.227273,0.0
4,network-device,Fisher-Weighted Z,device-type attack tuning,251,2.133570,0.000000,1.000000,1.000000,1.0,1.000000,1.0


,device_type,feature,fisher_weight,weight_source
28,camera,network_interval-packets,2.186254,device-type attack tuning
37,camera,network_ports_all_count,1.728203,device-type attack tuning
38,camera,network_ports_src_count,1.672671,device-type attack tuning
25,camera,network_packets_all_count,1.598572,device-type attack tuning
27,camera,network_packets_dst_count,1.530315,device-type attack tuning
71,edge-infrastructure,network_ttl_avg,7.938518,device-type attack tuning
72,edge-infrastructure,network_window-size_avg,6.864501,device-type attack tuning
59,edge-infrastructure,network_ips_all_count,0.932543,device-type attack tuning
61,edge-infrastructure,network_ips_dst_count,0.929682,device-type attack tuning
62,edge-infrastructure,network_ports_all_count,0.650315,device-type attack tuning


## 3. Define lightweight adaptive Fisher profile updates with rolling statistics

Each held-out client begins as a copy of its device-type profile. The online model then adapts only from accepted benign/drift evidence.

The implementation intentionally avoids storing complete accepted feature vectors. Enrollment, stable behavior, and drift candidates are summarized using a per-client rolling-statistics accumulator containing only the accepted-window count, per-feature running mean, and per-feature $M_2$ values. Drift confirmation additionally stores one scalar anomaly score per candidate window so the threshold can be recalibrated after confirmed drift.

The decision paths are:

- **Enrollment:** trusted benign windows update the initial client profile in fixed 64-window batches.
- **Stable:** low-scoring unflagged windows are accumulated and periodically blended into the client profile.
- **Drift candidate:** moderately elevated scores are accumulated until the PSO-selected confirmation length is reached; confirmed drift updates both profile statistics and the client threshold.
- **Hard anomaly / blocked anomaly:** suspicious windows are not used for adaptation, limiting self-poisoning.

The final binary decision uses a causal **2-of-3 evidence rule**: at least two of the most recent three raw anomaly flags must be positive.


In [3]:
# Fixed safety/decision controls. PSO tunes the seven parameters defined later
# (learning rates, anchoring, confirmation length, batch size, and threshold ceiling).
# Enrollment batch size remains fixed so all clients receive the same initial treatment.
ENROLLMENT_BATCH_SIZE = 64
RELATIVE_STD_FLOOR = 0.60
DRIFT_WARNING_RATIO = 0.80
HARD_SCORE_RATIO = 3.0
THRESHOLD_ALPHA = 0.20
UPDATE_CLIP_SIGMA = 3.0

# Final alarm smoothing: a window is confirmed when at least 2 of the most
# recent 3 raw flags are anomalous.
EVIDENCE_WINDOW = 3
EVIDENCE_MIN_HITS = 2

# Client enrollment threshold calibration controls.
CLIENT_THRESHOLD_PERCENTILE = 99.5
CLIENT_THRESHOLD_FLOOR_RATIO = 0.50


def new_client_profile(client_id, device_type):
    """Initialize a held-out client as a copy of its general device-type profile."""
    general = general_profiles[device_type]
    return ClientProfiles(
        client_id=client_id,
        device_type_name=device_type,
        feature_means=general.feature_means,
        feature_stds=general.feature_stds,
        feature_weights=general.feature_weights,
        threshold=general.threshold,
        std_floor=general.std_floor,
    )



def new_rolling_stats(score_capacity):
    """Create a compact rolling-statistics accumulator for one client.

    Only the accepted-window count, per-feature running mean, and per-feature
    running M2 values are retained. Drift confirmation additionally keeps one
    scalar anomaly score per candidate window for threshold recalibration.
    Complete feature vectors are not stored.
    """
    score_capacity = int(score_capacity)
    if score_capacity <= 0:
        raise ValueError("score_capacity must be positive.")

    return {
        "mode": None,
        "count": 0,
        "mean": np.zeros(len(FEATURE_COLUMNS), dtype=np.float64),
        "m2": np.zeros(len(FEATURE_COLUMNS), dtype=np.float64),
        "score_capacity": score_capacity,
        "scores": None,
    }


def reset_rolling_stats(stats):
    """Clear the current enrollment/stable/drift accumulator in place."""
    stats["mode"] = None
    stats["count"] = 0
    stats["mean"].fill(0.0)
    stats["m2"].fill(0.0)
    stats["scores"] = None


def new_state(score_capacity):
    """Create the compact online state retained for one adaptive client."""
    return {
        # Enrollment, stable, and drift accumulation are mutually exclusive,
        # so they reuse one per-client rolling-statistics accumulator.
        "rolling_stats": new_rolling_stats(score_capacity),
        "batch_updates": 0,
        "confirmed_drifts": 0,
        "blocked_extremes": 0,
    }


def effective_arrays(profile, params):
    """Blend client and general statistics according to the anchor strength."""
    client_means, client_stds = profile_arrays(profile)
    general_means, general_stds = profile_arrays(
        general_profiles[profile.device_type_name]
    )
    anchor = float(params["anchor_strength"])
    means = (1.0 - anchor) * client_means + anchor * general_means
    variances = (
        (1.0 - anchor) * client_stds**2
        + anchor * general_stds**2
        + anchor
        * (1.0 - anchor)
        * (client_means - general_means) ** 2
    )
    return means, np.sqrt(np.maximum(variances, SCORE_STD_FLOOR**2))


def score_general(profile, values):
    """Score one feature vector using a fixed general device-type profile."""
    window = dict(zip(FEATURE_COLUMNS, np.asarray(values, dtype=float)))
    return float(profile.score_window(window))


def score_client(profile, values, params):
    """Score one feature vector using the anchored adaptive client profile."""
    general = general_profiles[profile.device_type_name]
    window = dict(zip(FEATURE_COLUMNS, np.asarray(values, dtype=float)))
    return float(
        profile.score_window(
            window,
            general_profile=general,
            anchor_strength=float(params["anchor_strength"]),
        )
    )


def score_client_matrix(profile, values, params):
    """Vectorized adaptive scoring used during tuning and threshold calibration."""
    means, stds = effective_arrays(profile, params)
    return fisher_weighted_scores(
        values,
        means,
        stds,
        profile_weights(profile),
    )



def add_rolling_observation(
    stats,
    mode,
    values,
    profile,
    params,
    score=None,
    clip_values=True,
):
    """Update exact batch statistics without retaining the feature vector."""
    if stats["mode"] != mode:
        reset_rolling_stats(stats)
        stats["mode"] = mode

    observation = np.asarray(values, dtype=np.float64)
    expected_shape = (len(FEATURE_COLUMNS),)
    if observation.shape != expected_shape:
        raise ValueError(
            f"Expected {len(FEATURE_COLUMNS)} features, got {observation.shape}."
        )

    # The profile remains fixed while a batch is accumulating. Therefore,
    # clipping each accepted observation here is equivalent to clipping the
    # complete batch immediately before the former full-buffer update.
    if clip_values:
        live_means, live_stds = effective_arrays(profile, params)
        observation = np.clip(
            observation,
            live_means - UPDATE_CLIP_SIGMA * live_stds,
            live_means + UPDATE_CLIP_SIGMA * live_stds,
        )

    new_count = stats["count"] + 1
    delta = observation - stats["mean"]
    stats["mean"] += delta / new_count
    delta_after = observation - stats["mean"]
    stats["m2"] += delta * delta_after
    stats["count"] = new_count

    if mode == "drift":
        if score is None:
            raise ValueError("A scalar score is required for drift accumulation.")
        if stats["scores"] is None:
            stats["scores"] = np.empty(
                stats["score_capacity"],
                dtype=np.float64,
            )
        if new_count > stats["scores"].size:
            raise ValueError("Drift-score accumulator capacity was exceeded.")
        stats["scores"][new_count - 1] = float(score)

    return new_count


def update_profile_from_statistics(profile, stats, alpha):
    """Blend the client profile with the accumulated batch statistics."""
    if alpha <= 0.0 or stats["count"] == 0:
        return False

    batch_means = stats["mean"].copy()
    batch_variances = stats["m2"] / stats["count"]

    old_means, old_stds = profile_arrays(profile)
    new_means = (1.0 - alpha) * old_means + alpha * batch_means
    new_variances = (
        (1.0 - alpha) * old_stds**2
        + alpha * batch_variances
        + alpha * (1.0 - alpha) * (old_means - batch_means) ** 2
    )

    _, general_stds = profile_arrays(
        general_profiles[profile.device_type_name]
    )
    minimum_stds = np.maximum(
        SCORE_STD_FLOOR,
        RELATIVE_STD_FLOOR * general_stds,
    )
    new_stds = np.maximum(
        np.sqrt(np.maximum(new_variances, 0.0)),
        minimum_stds,
    )

    profile.feature_means = dict(zip(FEATURE_COLUMNS, new_means))
    profile.feature_stds = dict(zip(FEATURE_COLUMNS, new_stds))
    return True


def drift_evidence(profile, score):
    """Use only the Fisher score ratio to separate drift candidates from attacks."""
    safe = np.finfo(float).eps
    score_ratio = score / max(profile.threshold, safe)
    hard = score_ratio > HARD_SCORE_RATIO
    candidate = (
        not hard
        and score_ratio >= DRIFT_WARNING_RATIO
    )
    return candidate, hard, score_ratio



def recalibrate_threshold_from_scores(profile, scores, params):
    """Update the threshold from compact drift-candidate score statistics."""
    scores = np.asarray(scores, dtype=np.float64)
    scores = scores[np.isfinite(scores)]
    if scores.size == 0:
        return

    candidate = float(np.percentile(scores, SCORING_PERCENTILE))
    general_threshold = general_profiles[profile.device_type_name].threshold
    blended = (
        (1.0 - THRESHOLD_ALPHA) * profile.threshold
        + THRESHOLD_ALPHA * candidate
    )
    profile.threshold = float(
        np.clip(
            blended,
            general_threshold,
            float(params["threshold_ceiling"]) * general_threshold,
        )
    )



def process_observation(profile, values, params, state, trusted=False):
    """Score one window, then route it to a stable, drift, or blocked path."""
    score_start = time.perf_counter()
    score = score_client(profile, values, params)
    score_time_ms = (time.perf_counter() - score_start) * 1000
    threshold_before = float(profile.threshold)
    flagged = profile.is_anomalous(score)
    profile.window_count += 1

    candidate, hard, score_ratio = drift_evidence(profile, score)
    updated = False
    confirmed = False
    action = "no_update"
    update_start = time.perf_counter()
    stats = state["rolling_stats"]

    # Route the observation according to its trust/anomaly state. Only trusted,
    # stable, or confirmed-drift observations are eligible to change the profile.
    if trusted:
        # Enrollment is known to be benign. Preserve the former enrollment
        # behavior by accumulating its un-clipped feature statistics.
        count = add_rolling_observation(
            stats,
            "enrollment",
            values,
            profile,
            params,
            clip_values=False,
        )
        action = "enrollment_accumulator"
        if count >= ENROLLMENT_BATCH_SIZE:
            updated = update_profile_from_statistics(
                profile,
                stats,
                float(params["warmup_alpha"]),
            )
            reset_rolling_stats(stats)
            action = "enrollment_batch"

    elif hard:
        reset_rolling_stats(stats)
        state["blocked_extremes"] += 1
        action = "blocked_extreme"

    elif candidate:
        count = add_rolling_observation(
            stats,
            "drift",
            values,
            profile,
            params,
            score=score,
            clip_values=True,
        )
        action = "drift_accumulator"

        if count >= int(params["confirmation_windows"]):
            confirmed_scores = stats["scores"][:stats["count"]].copy()
            updated = update_profile_from_statistics(
                profile,
                stats,
                float(params["drift_alpha"]),
            )
            if updated:
                # Complete drift feature vectors are not retained. The compact
                # score sequence observed during confirmation is therefore used
                # for percentile-based threshold recalibration.
                recalibrate_threshold_from_scores(
                    profile,
                    confirmed_scores,
                    params,
                )
                state["confirmed_drifts"] += 1
                confirmed = True

            reset_rolling_stats(stats)
            action = "drift_confirmed"

    elif not flagged:
        count = add_rolling_observation(
            stats,
            "stable",
            values,
            profile,
            params,
            clip_values=True,
        )
        action = "stable_accumulator"

        if count >= int(params["stable_batch_size"]):
            updated = update_profile_from_statistics(
                profile,
                stats,
                float(params["stable_alpha"]),
            )
            reset_rolling_stats(stats)
            action = "stable_batch"

    else:
        reset_rolling_stats(stats)
        action = "blocked_anomaly"

    if updated:
        state["batch_updates"] += 1

    return {
        "score": score,
        "threshold_before": threshold_before,
        "threshold_after": float(profile.threshold),
        "flagged": bool(flagged),
        "updated": bool(updated),
        "update_action": action,
        "drift_confirmed": bool(confirmed),
        "score_ratio": float(score_ratio),
        "score_time_ms": float(score_time_ms),
        "update_time_ms": float((time.perf_counter() - update_start) * 1000),
    }



def flush_enrollment(profile, params, state):
    """Apply any partial trusted-enrollment batch left after full batches."""
    stats = state["rolling_stats"]
    if stats["mode"] != "enrollment" or stats["count"] == 0:
        return False

    updated = update_profile_from_statistics(
        profile,
        stats,
        float(params["warmup_alpha"]),
    )
    reset_rolling_stats(stats)
    state["batch_updates"] += int(updated)
    return updated


def calibrate_client_threshold(profile, enrollment_values, params):
    """Calibrate a bounded client threshold from trusted enrollment only."""
    scores = score_client_matrix(
        profile,
        np.asarray(enrollment_values, dtype=float),
        params,
    )
    general_threshold = general_profiles[
        profile.device_type_name
    ].threshold
    candidate = float(
        np.percentile(scores, CLIENT_THRESHOLD_PERCENTILE)
    )
    profile.threshold = float(
        np.clip(
            candidate,
            CLIENT_THRESHOLD_FLOOR_RATIO * general_threshold,
            general_threshold,
        )
    )
    return profile.threshold


def controlled_drift(enrollment_values, tuning_values, general_stds):
    """Create a moderate broad shift used only while tuning adaptation."""
    direction = np.sign(
        np.median(tuning_values, axis=0)
        - np.median(enrollment_values, axis=0)
    )
    fallback = np.where(
        np.arange(tuning_values.shape[1]) % 2 == 0,
        1.0,
        -1.0,
    )
    direction = np.where(direction == 0.0, fallback, direction)
    return tuning_values + 0.85 * general_stds * direction


def add_evidence_flags(dataframe, group_columns, order_column):
    """Apply the causal final decision: at least two raw flags in the last three windows."""
    result = dataframe.copy()
    result["evidence_flagged"] = False
    for _, group in result.groupby(group_columns, sort=False):
        ordered = group.sort_values(order_column, kind="stable")
        hits = (
            ordered["flagged"]
            .astype(int)
            .rolling(EVIDENCE_WINDOW, min_periods=1)
            .sum()
        )
        result.loc[ordered.index, "evidence_flagged"] = (
            hits >= EVIDENCE_MIN_HITS
        ).to_numpy()
    return result


## 4. Tune the seven adaptation parameters with PSO

PSO is run independently for each general device type. A candidate parameter set is evaluated only on held-out enrollment/tuning data, controlled synthetic drift derived from the tuning data, and the disjoint attack-tuning split.

The objective prioritizes low benign false-positive rates while enforcing a security constraint: adaptive attack detection for each client and attack type should remain within one percentage point of the corresponding static detector. If no fully feasible candidate is found, the lowest-shortfall fallback is retained.


In [4]:
# Cache each held-out client partition as NumPy arrays so PSO can repeatedly
# evaluate candidates without rebuilding the same DataFrame slices.
ENROLLMENT_BY_CLIENT = {
    client: rows.sort_values("position", kind="stable")[
        FEATURE_COLUMNS
    ].to_numpy(dtype=float)
    for client, rows in enrollment_df.groupby("device_name", sort=False)
}
TUNING_BY_CLIENT = {
    client: rows.sort_values("position", kind="stable")[
        FEATURE_COLUMNS
    ].to_numpy(dtype=float)
    for client, rows in tuning_df.groupby("device_name", sort=False)
}
TYPE_BY_CLIENT = (
    enrollment_df[["device_name", "device_type"]]
    .drop_duplicates()
    .set_index("device_name")["device_type"]
    .to_dict()
)
CLIENTS_BY_TYPE = {
    device_type: sorted(
        client
        for client, client_type in TYPE_BY_CLIENT.items()
        if client_type == device_type
    )
    for device_type in general_profiles
}
ATTACK_TUNING_ROWS_BY_CLIENT = {
    client: rows.copy()
    for client, rows in attack_tuning_df.groupby(
        "device_name",
        sort=False,
    )
}


def attack_checkpoint_rows(
    profile,
    client_id,
    device_type,
    params,
    checkpoint,
):
    """Score one client's tuning attacks at one adaptation checkpoint."""
    rows = ATTACK_TUNING_ROWS_BY_CLIENT[client_id]
    values = rows[FEATURE_COLUMNS].to_numpy(dtype=float)
    adaptive_scores = score_client_matrix(profile, values, params)

    general = general_profiles[device_type]
    general_means, general_stds = profile_arrays(general)
    static_scores = fisher_weighted_scores(
        values,
        general_means,
        general_stds,
        profile_weights(general),
    )

    return pd.DataFrame({
        "checkpoint": checkpoint,
        "device_name": client_id,
        "attack_type": rows["attack_type"].to_numpy(),
        "adaptive_flagged": adaptive_scores > profile.threshold,
        "static_flagged": static_scores > general.threshold,
    })


def detection_constraint_rows(checkpoint_df, group_column):
    """Compare adaptive detection with static detection for one grouping view."""
    grouped = checkpoint_df.groupby(
        ["checkpoint", group_column],
        sort=False,
    ).agg(
        detection=("adaptive_flagged", "mean"),
        static_detection=("static_flagged", "mean"),
        tuning_windows=("adaptive_flagged", "size"),
    ).reset_index()
    grouped["group_view"] = group_column
    grouped["group_name"] = grouped[group_column].astype(str)
    grouped["minimum_detection"] = np.maximum(
        0.0,
        grouped["static_detection"] - 0.01,
    )
    grouped["shortfall"] = np.maximum(
        0.0,
        grouped["minimum_detection"] - grouped["detection"],
    )
    return grouped[
        [
            "checkpoint",
            "group_view",
            "group_name",
            "tuning_windows",
            "detection",
            "static_detection",
            "minimum_detection",
            "shortfall",
        ]
    ]


def simulate_candidate(device_type, params):
    """Evaluate FPR and security constraints without final-test leakage."""
    stable_flags = []
    drift_flags = []
    stable_fpr_by_client = []
    drift_fpr_by_client = []
    attack_checkpoints = []

    # Recreate each client from the same initial general profile for every
    # candidate so PSO comparisons are fair and independent.
    for client_id in CLIENTS_BY_TYPE[device_type]:
        profile = new_client_profile(client_id, device_type)
        state = new_state(int(params["confirmation_windows"]))
        enrollment_values = ENROLLMENT_BY_CLIENT[client_id]
        tuning_values = TUNING_BY_CLIENT[client_id]

        for values in enrollment_values:
            process_observation(
                profile,
                values,
                params,
                state,
                trusted=True,
            )
        flush_enrollment(profile, params, state)
        calibrate_client_threshold(
            profile,
            enrollment_values,
            params,
        )
        attack_checkpoints.append(
            attack_checkpoint_rows(
                profile,
                client_id,
                device_type,
                params,
                "after_enrollment",
            )
        )

        client_stable_flags = []
        for values in tuning_values:
            client_stable_flags.append(
                process_observation(
                    profile,
                    values,
                    params,
                    state,
                )["flagged"]
            )
        stable_flags.extend(client_stable_flags)
        stable_fpr_by_client.append(
            float(np.mean(client_stable_flags))
        )
        attack_checkpoints.append(
            attack_checkpoint_rows(
                profile,
                client_id,
                device_type,
                params,
                "after_benign_tuning",
            )
        )

        _, general_stds = profile_arrays(general_profiles[device_type])
        shifted_values = controlled_drift(
            enrollment_values,
            tuning_values,
            general_stds,
        )
        client_drift_flags = []
        for values in shifted_values:
            client_drift_flags.append(
                process_observation(
                    profile,
                    values,
                    params,
                    state,
                )["flagged"]
            )
        drift_flags.extend(client_drift_flags)
        drift_fpr_by_client.append(
            float(np.mean(client_drift_flags))
        )
        attack_checkpoints.append(
            attack_checkpoint_rows(
                profile,
                client_id,
                device_type,
                params,
                "after_controlled_drift",
            )
        )

    checkpoint_df = pd.concat(
        attack_checkpoints,
        ignore_index=True,
    )
    constraint_df = pd.concat(
        [
            detection_constraint_rows(checkpoint_df, "device_name"),
            detection_constraint_rows(checkpoint_df, "attack_type"),
        ],
        ignore_index=True,
    )

    stable_fpr = float(np.mean(stable_flags))
    drift_fpr = float(np.mean(drift_flags))
    worst_client_stable_fpr = float(max(stable_fpr_by_client))
    worst_client_drift_fpr = float(max(drift_fpr_by_client))
    detection_rate = float(checkpoint_df["adaptive_flagged"].mean())
    worst_group_detection = float(constraint_df["detection"].min())
    mean_group_shortfall = float(constraint_df["shortfall"].mean())
    maximum_group_shortfall = float(constraint_df["shortfall"].max())
    feasible = maximum_group_shortfall <= 1e-12

    # Security constraint: adaptive detection must not materially degrade
    # relative to the static detector for any evaluated client/attack-type view.
    # Infeasible candidates cannot win because the million-point barrier
    # dominates every FPR term. The smaller terms rank feasible candidates.
    objective = (
        1_000_000.0 * float(not feasible)
        + 4.0 * stable_fpr
        + 2.0 * drift_fpr
        + 2.0 * worst_client_stable_fpr
        + 1.0 * worst_client_drift_fpr
        + 50.0 * mean_group_shortfall
        + 100.0 * maximum_group_shortfall
        + 0.05 * (1.0 - detection_rate)
    )
    return {
        "objective": objective,
        "feasible": bool(feasible),
        "stable_fpr": stable_fpr,
        "controlled_drift_fpr": drift_fpr,
        "worst_client_stable_fpr": worst_client_stable_fpr,
        "worst_client_controlled_drift_fpr": worst_client_drift_fpr,
        "detection_rate": detection_rate,
        "worst_group_detection": worst_group_detection,
        "mean_group_shortfall": mean_group_shortfall,
        "maximum_group_shortfall": maximum_group_shortfall,
    }


In [5]:
PSO_SEED = 42
PSO_PARTICLES = 8
PSO_ITERATIONS = 10
PSO_INERTIA = 0.70
PSO_COGNITIVE = 1.50
PSO_SOCIAL = 1.50

# Seven online-adaptation parameters optimized independently per device type.
PARAMETER_NAMES = [
    "warmup_alpha",
    "stable_alpha",
    "drift_alpha",
    "anchor_strength",
    "confirmation_windows",
    "stable_batch_size",
    "threshold_ceiling",
]


def interpolate(value, lower, upper):
    """Map a normalized particle coordinate from [0, 1] to a linear range."""
    return lower + value * (upper - lower)


def log_interpolate(value, lower, upper):
    """Map a normalized coordinate to a log-scaled positive parameter range."""
    return 10 ** interpolate(value, np.log10(lower), np.log10(upper))


def decode_particle(position):
    """Convert one normalized PSO particle into framework hyperparameters."""
    return {
        "warmup_alpha": log_interpolate(position[0], 0.050, 0.750),
        "stable_alpha": log_interpolate(position[1], 0.00001, 0.00500),
        "drift_alpha": log_interpolate(position[2], 0.010, 0.250),
        "anchor_strength": interpolate(position[3], 0.05, 0.60),
        "confirmation_windows": int(round(interpolate(position[4], 8, 64))),
        "stable_batch_size": int(round(interpolate(position[5], 16, 128))),
        "threshold_ceiling": interpolate(position[6], 1.00, 1.60),
    }


def conservative_params():
    """Return a hand-set safe candidate that PSO must outperform to be selected."""
    return {
        "warmup_alpha": 0.250,
        "stable_alpha": 0.00001,
        "drift_alpha": 0.050,
        "anchor_strength": 0.25,
        "confirmation_windows": 32,
        "stable_batch_size": 64,
        "threshold_ceiling": 1.30,
    }


def run_pso(device_type, seed):
    """Tune one device type with standard particle-swarm position/velocity updates."""
    rng = np.random.default_rng(seed)
    dimensions = len(PARAMETER_NAMES)
    positions = rng.uniform(0.0, 1.0, size=(PSO_PARTICLES, dimensions))
    velocities = rng.uniform(-0.10, 0.10, size=(PSO_PARTICLES, dimensions))
    personal_best_positions = positions.copy()
    personal_best_scores = np.full(PSO_PARTICLES, np.inf)
    global_best_position = None
    global_best_score = np.inf
    global_best_metrics = None
    history = []

    baseline_params = conservative_params()
    baseline_metrics = simulate_candidate(device_type, baseline_params)
    history.append({
        "device_type": device_type,
        "iteration": 0,
        "particle": 0,
        "candidate": "conservative",
        **baseline_params,
        **baseline_metrics,
    })

    for iteration in range(1, PSO_ITERATIONS + 1):
        for particle_index in range(PSO_PARTICLES):
            params = decode_particle(positions[particle_index])
            metrics = simulate_candidate(device_type, params)
            history.append({
                "device_type": device_type,
                "iteration": iteration,
                "particle": particle_index + 1,
                "candidate": "pso",
                **params,
                **metrics,
            })

            if metrics["objective"] < personal_best_scores[particle_index]:
                personal_best_scores[particle_index] = metrics["objective"]
                personal_best_positions[particle_index] = positions[
                    particle_index
                ].copy()

            if metrics["objective"] < global_best_score:
                global_best_score = metrics["objective"]
                global_best_position = positions[particle_index].copy()
                global_best_metrics = metrics.copy()

        print(
            f"{device_type:>20} | iteration {iteration:>2}/{PSO_ITERATIONS} | "
            f"objective={global_best_score:.6f} | "
            f"stable FPR={global_best_metrics['stable_fpr']:.3%} | "
            f"drift FPR={global_best_metrics['controlled_drift_fpr']:.3%} | "
            f"worst attack={global_best_metrics['worst_group_detection']:.3%}"
        )

        # Standard PSO update: inertia keeps momentum, while cognitive and
        # social terms pull each particle toward its personal and global bests.
        r1 = rng.random(size=(PSO_PARTICLES, dimensions))
        r2 = rng.random(size=(PSO_PARTICLES, dimensions))
        velocities = (
            PSO_INERTIA * velocities
            + PSO_COGNITIVE
            * r1
            * (personal_best_positions - positions)
            + PSO_SOCIAL
            * r2
            * (global_best_position - positions)
        )
        positions = np.clip(positions + velocities, 0.0, 1.0)

    if baseline_metrics["objective"] <= global_best_score:
        selected_params = baseline_params
        selected_metrics = baseline_metrics
        selected_from = "conservative"
    else:
        selected_params = decode_particle(global_best_position)
        selected_metrics = global_best_metrics
        selected_from = "pso"

    if not selected_metrics["feasible"]:
        selected_from += "_minimum_shortfall_fallback"
        print(
            f"{device_type:>20} | no fully feasible low-FPR candidate; "
            "using the minimum-shortfall fallback"
        )

    return (
        selected_params,
        selected_metrics,
        selected_from,
        pd.DataFrame(history),
    )


## 5. Calibrate held-out client profiles

Using the best PSO parameters for each device type, this section creates one adaptive profile per held-out client. Each client is first updated from its trusted enrollment partition, receives a bounded client-specific threshold, and is then adapted on the benign tuning stream.

These calibrated profiles are the starting point for the untouched final streaming evaluation in the next section.


In [6]:
# Run PSO once per general device type and retain the selected parameters.
BEST_PARAMS_BY_TYPE = {}
pso_rows = []
pso_histories = []

for type_index, device_type in enumerate(general_profiles):
    params, metrics, selected_from, history = run_pso(
        device_type,
        PSO_SEED + type_index,
    )
    BEST_PARAMS_BY_TYPE[device_type] = params
    pso_rows.append({
        "device_type": device_type,
        "selected_from": selected_from,
        **params,
        **metrics,
    })
    pso_histories.append(history)

pso_summary = pd.DataFrame(pso_rows)
pso_history = pd.concat(pso_histories, ignore_index=True)
display(pso_summary)


def calibrate_clients():
    """Enroll each held-out client, then adapt it on the benign tuning stream."""
    client_profiles = {}
    client_states = {}

    for client_id, enrollment_values in ENROLLMENT_BY_CLIENT.items():
        device_type = TYPE_BY_CLIENT[client_id]
        params = BEST_PARAMS_BY_TYPE[device_type]
        profile = new_client_profile(client_id, device_type)
        state = new_state(int(params["confirmation_windows"]))

        for values in enrollment_values:
            process_observation(
                profile,
                values,
                params,
                state,
                trusted=True,
            )
        flush_enrollment(profile, params, state)
        calibrate_client_threshold(
            profile,
            enrollment_values,
            params,
        )

        for values in TUNING_BY_CLIENT[client_id]:
            process_observation(
                profile,
                values,
                params,
                state,
            )

        client_profiles[client_id] = profile
        client_states[client_id] = state

    return client_profiles, client_states


# These profiles/states are now frozen as the starting point of final testing.
calibrated_profiles, calibrated_states = calibrate_clients()


              sensor | iteration  1/10 | objective=1000006.362461 | stable FPR=0.000% | drift FPR=0.198% | worst attack=0.000%
              sensor | iteration  2/10 | objective=1000006.344604 | stable FPR=0.000% | drift FPR=0.000% | worst attack=0.000%
              sensor | iteration  3/10 | objective=1000006.344604 | stable FPR=0.000% | drift FPR=0.000% | worst attack=0.000%
              sensor | iteration  4/10 | objective=1000006.344604 | stable FPR=0.000% | drift FPR=0.000% | worst attack=0.000%
              sensor | iteration  5/10 | objective=1000006.344604 | stable FPR=0.000% | drift FPR=0.000% | worst attack=0.000%
              sensor | iteration  6/10 | objective=1000006.344604 | stable FPR=0.000% | drift FPR=0.000% | worst attack=0.000%
              sensor | iteration  7/10 | objective=1000006.344604 | stable FPR=0.000% | drift FPR=0.000% | worst attack=0.000%
              sensor | iteration  8/10 | objective=1000006.344604 | stable FPR=0.000% | drift FPR=0.000% | wors

,device_type,selected_from,warmup_alpha,stable_alpha,drift_alpha,anchor_strength,confirmation_windows,stable_batch_size,threshold_ceiling,objective,feasible,stable_fpr,controlled_drift_fpr,worst_client_stable_fpr,worst_client_controlled_drift_fpr,detection_rate,worst_group_detection,mean_group_shortfall,maximum_group_shortfall
0,sensor,pso_minimum_shortfall_fallback,0.463582,0.000088,0.250000,0.498734,8,128,1.438791,1.000006e+06,False,0.000000,0.000000,0.000000,0.000000,0.878641,0.0,0.003914,0.061429
1,camera,pso,0.091313,0.000129,0.047754,0.373602,8,81,1.088464,3.004481e-01,True,0.009259,0.046296,0.027778,0.111111,0.916963,0.0,0.000000,0.000000
2,edge-infrastructure,pso_minimum_shortfall_fallback,0.254892,0.000011,0.111182,0.550749,36,22,1.105950,1.000107e+06,False,0.000000,0.000000,0.000000,0.000000,0.759878,0.0,0.152004,0.990000
3,smart-plug,pso_minimum_shortfall_fallback,0.069193,0.000766,0.010108,0.422298,35,16,1.153971,1.000006e+06,False,0.001984,0.013889,0.013889,0.013889,0.327586,0.0,0.003724,0.061429
4,network-device,pso_minimum_shortfall_fallback,0.750000,0.000485,0.043146,0.092088,48,35,1.323240,1.000105e+06,False,0.006944,0.006944,0.013889,0.013889,0.862245,0.0,0.124504,0.990000


## 6. Run static, drift, adaptive, and attack evaluations

Three benign conditions are compared:

- **Baseline:** the static general profile on known-device baseline data.
- **Drift Simulation:** the same static general profile on held-out client traffic, showing the effect of behavioral heterogeneity/drift without adaptation.
- **Adaptive Fisher-Weighted Z:** the client-specific adaptive profile on the same held-out final stream.

After the benign stream has been processed, the untouched attack-test partition is scored by both the static general profiles and the final adaptive client profiles. Attack rows are scored only; they are not used to update the adaptive profiles.


In [7]:
def run_static_benign(dataframe, model_name):
    """Score benign windows with fixed general profiles and apply 2-of-3 evidence."""
    rows_out = []
    ordered = dataframe.sort_values(
        ["device_name", "position"],
        kind="stable",
    )
    for _, row in ordered.iterrows():
        start = time.perf_counter()
        profile = general_profiles[row["device_type"]]
        values = row[FEATURE_COLUMNS].to_numpy(dtype=float)
        score = score_general(profile, values)
        rows_out.append({
            "model": model_name,
            "device_name": row["device_name"],
            "device_type": row["device_type"],
            "position": int(row["position"]),
            "score": score,
            "threshold": profile.threshold,
            "flagged": profile.is_anomalous(score),
            "total_time_ms": (time.perf_counter() - start) * 1000,
        })
    return add_evidence_flags(
        pd.DataFrame(rows_out),
        group_columns=["device_name"],
        order_column="position",
    )


def run_adaptive_benign():
    """Process the untouched held-out benign stream with online client adaptation."""
    rows_out = []
    ordered = stream_df.sort_values(
        ["device_name", "position"],
        kind="stable",
    )
    for _, row in ordered.iterrows():
        start = time.perf_counter()
        client_id = row["device_name"]
        device_type = row["device_type"]
        profile = calibrated_profiles[client_id]
        state = calibrated_states[client_id]
        params = BEST_PARAMS_BY_TYPE[device_type]
        values = row[FEATURE_COLUMNS].to_numpy(dtype=float)
        adaptive = process_observation(profile, values, params, state)

        rows_out.append({
            "model": "Adaptive Fisher-Weighted Z",
            "device_name": client_id,
            "device_type": device_type,
            "position": int(row["position"]),
            "score": adaptive["score"],
            "threshold": adaptive["threshold_before"],
            "flagged": bool(adaptive["flagged"]),
            "updated": adaptive["updated"],
            "update_action": adaptive["update_action"],
            "drift_confirmed": adaptive["drift_confirmed"],
            "total_time_ms": (time.perf_counter() - start) * 1000,
        })
    return add_evidence_flags(
        pd.DataFrame(rows_out),
        group_columns=["device_name"],
        order_column="position",
    )


# The adaptive benign run intentionally mutates the calibrated client profiles
# as the final stream arrives; attack scoring therefore uses their final online state.
baseline_results = run_static_benign(baseline_test_df, "Baseline")
drift_results = run_static_benign(stream_df, "Drift Simulation")
adaptive_results = run_adaptive_benign()


ATTACK_GROUP_COLUMNS = [
    "device_name",
    "attack_family",
    "attack_type",
]


def run_static_attacks():
    """Score untouched attack-test windows with the fixed general profiles."""
    rows_out = []
    for _, row in attack_test_df.iterrows():
        profile = general_profiles[row["device_type"]]
        values = row[FEATURE_COLUMNS].to_numpy(dtype=float)
        score = score_general(profile, values)
        rows_out.append({
            "model": "Drift Simulation",
            "device_name": row["device_name"],
            "device_type": row["device_type"],
            "attack_family": row["attack_family"],
            "attack_type": row["attack_type"],
            "attack_position": int(row["attack_position"]),
            "flagged": profile.is_anomalous(score),
        })
    return add_evidence_flags(
        pd.DataFrame(rows_out),
        group_columns=ATTACK_GROUP_COLUMNS,
        order_column="attack_position",
    )


def run_adaptive_attacks():
    """Score attacks using final adaptive profiles without adapting on attack rows."""
    rows_out = []
    for _, row in attack_test_df.iterrows():
        client_id = row["device_name"]
        device_type = row["device_type"]
        profile = calibrated_profiles[client_id]
        params = BEST_PARAMS_BY_TYPE[device_type]
        values = row[FEATURE_COLUMNS].to_numpy(dtype=float)
        adaptive_score = score_client(profile, values, params)

        rows_out.append({
            "model": "Adaptive Fisher-Weighted Z",
            "device_name": client_id,
            "device_type": device_type,
            "attack_family": row["attack_family"],
            "attack_type": row["attack_type"],
            "attack_position": int(row["attack_position"]),
            "flagged": profile.is_anomalous(adaptive_score),
        })
    return add_evidence_flags(
        pd.DataFrame(rows_out),
        group_columns=ATTACK_GROUP_COLUMNS,
        order_column="attack_position",
    )


static_attack_results = run_static_attacks()
adaptive_attack_results = run_adaptive_attacks()


## 7. Report and save final evidence-level results

This section converts raw window flags into the 2-of-3 evidence decisions used in the paper, summarizes benign FPR and attack detection at both window and event levels, and reports runtime/storage overhead.

The final CSV tables are written to `results_datasense_rolling_stats/` so the reported values can be inspected without rerunning PSO.


In [8]:
def raw_fpr(results):
    """Return the per-window false-positive rate before evidence smoothing."""
    return float(results["flagged"].mean())


def confirmed_fpr(results):
    """Return the final false-positive rate after the 2-of-3 evidence rule."""
    return float(results["evidence_flagged"].mean())


def summarize_attack_groups(results):
    """Summarize raw/confirmed detection for each device-family-type attack event."""
    rows_out = []
    for group_key, rows in results.groupby(
        ATTACK_GROUP_COLUMNS,
        sort=False,
    ):
        ordered = rows.sort_values(
            "attack_position",
            kind="stable",
        )
        confirmed = ordered["evidence_flagged"].to_numpy(dtype=bool)
        event_detected = bool(confirmed.any())
        confirmation_delay = (
            int(np.flatnonzero(confirmed)[0] + 1)
            if event_detected
            else np.nan
        )
        rows_out.append({
            "model": ordered["model"].iloc[0],
            "device_name": group_key[0],
            "device_type": ordered["device_type"].iloc[0],
            "attack_family": group_key[1],
            "attack_type": group_key[2],
            "raw_detected_windows": int(ordered["flagged"].sum()),
            "confirmed_windows": int(confirmed.sum()),
            "total_windows": len(ordered),
            "event_detected": event_detected,
            "confirmation_delay_windows": confirmation_delay,
        })
    summary = pd.DataFrame(rows_out)
    summary["raw_detection_rate"] = (
        summary["raw_detected_windows"]
        / summary["total_windows"]
    )
    summary["confirmed_detection_rate"] = (
        summary["confirmed_windows"]
        / summary["total_windows"]
    )
    return summary


def overall_attack_metrics(group_summary):
    """Aggregate attack-window and attack-event detection statistics."""
    total_windows = group_summary["total_windows"].sum()
    detected_events = int(group_summary["event_detected"].sum())
    total_events = len(group_summary)
    return {
        "attack_windows": int(total_windows),
        "raw_detection_rate": float(
            group_summary["raw_detected_windows"].sum()
            / total_windows
        ),
        "confirmed_detection_rate": float(
            group_summary["confirmed_windows"].sum()
            / total_windows
        ),
        "attack_events": total_events,
        "detected_events": detected_events,
        "event_detection_rate": float(
            detected_events / total_events
        ),
        "mean_confirmation_delay_windows": float(
            group_summary.loc[
                group_summary["event_detected"],
                "confirmation_delay_windows",
            ].mean()
        ),
    }


static_attack_group_results = summarize_attack_groups(
    static_attack_results
)
adaptive_attack_group_results = summarize_attack_groups(
    adaptive_attack_results
)
static_detection = overall_attack_metrics(
    static_attack_group_results
)
adaptive_detection = overall_attack_metrics(
    adaptive_attack_group_results
)

overall_results = pd.DataFrame([
    {
        "dataset": "CIC DataSense IIoT 2025",
        "experiment": "Baseline",
        "benign_windows": len(baseline_results),
        "raw_fpr": raw_fpr(baseline_results),
        "fpr": confirmed_fpr(baseline_results),
        "attack_windows": np.nan,
        "raw_detection_rate": np.nan,
        "detection_rate": np.nan,
        "attack_events": np.nan,
        "event_detection_rate": np.nan,
        "mean_confirmation_delay_windows": np.nan,
    },
    {
        "dataset": "CIC DataSense IIoT 2025",
        "experiment": "Drift Simulation",
        "benign_windows": len(drift_results),
        "raw_fpr": raw_fpr(drift_results),
        "fpr": confirmed_fpr(drift_results),
        **static_detection,
        "detection_rate": static_detection[
            "confirmed_detection_rate"
        ],
    },
    {
        "dataset": "CIC DataSense IIoT 2025",
        "experiment": "Adaptive Fisher-Weighted Z",
        "benign_windows": len(adaptive_results),
        "raw_fpr": raw_fpr(adaptive_results),
        "fpr": confirmed_fpr(adaptive_results),
        **adaptive_detection,
        "detection_rate": adaptive_detection[
            "confirmed_detection_rate"
        ],
    },
])

all_benign_results = pd.concat(
    [baseline_results, drift_results, adaptive_results],
    ignore_index=True,
)
device_type_fpr = all_benign_results.groupby(
    ["model", "device_type"],
    sort=False,
).agg(
    raw_false_flags=("flagged", "sum"),
    confirmed_false_flags=("evidence_flagged", "sum"),
    total_windows=("flagged", "size"),
    raw_fpr=("flagged", "mean"),
    fpr=("evidence_flagged", "mean"),
).reset_index()

device_fpr = all_benign_results.groupby(
    ["model", "device_name", "device_type"],
    sort=False,
).agg(
    raw_false_flags=("flagged", "sum"),
    confirmed_false_flags=("evidence_flagged", "sum"),
    total_windows=("flagged", "size"),
    raw_fpr=("flagged", "mean"),
    fpr=("evidence_flagged", "mean"),
).reset_index()

attack_group_detection = pd.concat(
    [static_attack_group_results, adaptive_attack_group_results],
    ignore_index=True,
)
attack_type_detection = attack_group_detection.groupby(
    ["model", "attack_family", "attack_type"],
    sort=False,
).agg(
    raw_detected_windows=("raw_detected_windows", "sum"),
    confirmed_windows=("confirmed_windows", "sum"),
    total_windows=("total_windows", "sum"),
    detected_events=("event_detected", "sum"),
    total_events=("event_detected", "size"),
).reset_index()
attack_type_detection["raw_detection_rate"] = (
    attack_type_detection["raw_detected_windows"]
    / attack_type_detection["total_windows"]
)
attack_type_detection["detection_rate"] = (
    attack_type_detection["confirmed_windows"]
    / attack_type_detection["total_windows"]
)
attack_type_detection["event_detection_rate"] = (
    attack_type_detection["detected_events"]
    / attack_type_detection["total_events"]
)

runtime_summary = all_benign_results.groupby(
    "model",
    sort=False,
)["total_time_ms"].agg(
    average_ms="mean",
    median_ms="median",
    maximum_ms="max",
).reset_index()

FLOAT64_BYTES = np.dtype(np.float64).itemsize
PROFILE_NUMERIC_VALUES = 3 * len(FEATURE_COLUMNS) + 1
PROFILE_NUMERIC_BYTES = PROFILE_NUMERIC_VALUES * FLOAT64_BYTES

# Raw numeric state only. Python object and dictionary overhead is excluded.
ROLLING_STATS_BYTES = (
    1 + 2 * len(FEATURE_COLUMNS)
) * FLOAT64_BYTES
MAX_CONFIRMATION_WINDOWS = max(
    int(params["confirmation_windows"])
    for params in BEST_PARAMS_BY_TYPE.values()
)
PEAK_DRIFT_STATE_BYTES = (
    ROLLING_STATS_BYTES
    + MAX_CONFIRMATION_WINDOWS * FLOAT64_BYTES
)

storage_summary = pd.DataFrame([
    {
        "storage_scope": "general profile statistical state",
        "items": len(general_profiles),
        "bytes_per_item": PROFILE_NUMERIC_BYTES,
        "total_bytes": len(general_profiles) * PROFILE_NUMERIC_BYTES,
        "details": (
            f"{len(FEATURE_COLUMNS)} means + "
            f"{len(FEATURE_COLUMNS)} standard deviations + "
            f"{len(FEATURE_COLUMNS)} Fisher weights + 1 threshold"
        ),
    },
    {
        "storage_scope": "client profile statistical state",
        "items": len(calibrated_profiles),
        "bytes_per_item": PROFILE_NUMERIC_BYTES,
        "total_bytes": len(calibrated_profiles) * PROFILE_NUMERIC_BYTES,
        "details": (
            f"{len(FEATURE_COLUMNS)} means + "
            f"{len(FEATURE_COLUMNS)} standard deviations + "
            f"{len(FEATURE_COLUMNS)} Fisher weights + 1 threshold"
        ),
    },
    {
        "storage_scope": "stable rolling-statistics state",
        "items": len(calibrated_profiles),
        "bytes_per_item": ROLLING_STATS_BYTES,
        "total_bytes": len(calibrated_profiles) * ROLLING_STATS_BYTES,
        "details": (
            f"count + {len(FEATURE_COLUMNS)} running means + "
            f"{len(FEATURE_COLUMNS)} running M2 values; "
            "stable batch closes at a PSO-selected size of "
            f"{min(int(p['stable_batch_size']) for p in BEST_PARAMS_BY_TYPE.values())}-"
            f"{max(int(p['stable_batch_size']) for p in BEST_PARAMS_BY_TYPE.values())} windows across device types"
        ),
    },
    {
        "storage_scope": "peak drift rolling-statistics state",
        "items": len(calibrated_profiles),
        "bytes_per_item": PEAK_DRIFT_STATE_BYTES,
        "total_bytes": len(calibrated_profiles) * PEAK_DRIFT_STATE_BYTES,
        "details": (
            f"rolling statistics + up to {MAX_CONFIRMATION_WINDOWS} "
            "scalar drift scores; no complete feature vectors retained"
        ),
    },
])

outputs = {
    "overall_results.csv": overall_results,
    "scoring_summary.csv": general_profile_summary,
    "fisher_feature_weights.csv": fisher_feature_weights,
    "device_type_fpr.csv": device_type_fpr,
    "device_fpr.csv": device_fpr,
    "attack_type_detection.csv": attack_type_detection,
    "attack_group_detection.csv": attack_group_detection,
    "pso_summary.csv": pso_summary,
    "pso_history.csv": pso_history,
    "runtime_summary.csv": runtime_summary,
    "storage_summary.csv": storage_summary,
    "split_summary.csv": split_summary,
}
for filename, dataframe in outputs.items():
    dataframe.to_csv(RESULTS_DIR / filename, index=False)

print(
    "Adaptive Fisher-Weighted Z uses the deployed 2-of-3 decision. Detection is reported as "
    "raw-window, confirmed-window, attack-event, and confirmation delay."
)
display(overall_results)
display(device_type_fpr)
display(attack_type_detection)
display(runtime_summary)
display(storage_summary)


Adaptive Fisher-Weighted Z uses the deployed 2-of-3 decision. Detection is reported as raw-window, confirmed-window, attack-event, and confirmation delay.


,dataset,experiment,benign_windows,raw_fpr,fpr,attack_windows,raw_detection_rate,detection_rate,attack_events,event_detection_rate,mean_confirmation_delay_windows,confirmed_detection_rate,detected_events
0,CIC DataSense IIoT 2025,Baseline,1296,0.010802,0.003086,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CIC DataSense IIoT 2025,Drift Simulation,4320,0.161343,0.151389,6858.0,0.872558,0.805628,546.0,0.888278,2.344330,0.805628,485.0
2,CIC DataSense IIoT 2025,Adaptive Fisher-Weighted Z,4320,0.009259,0.001852,6858.0,0.809128,0.747011,546.0,0.869963,2.484211,0.747011,475.0


,model,device_type,raw_false_flags,confirmed_false_flags,total_windows,raw_fpr,fpr
0,Baseline,sensor,5,4,504,0.009921,0.007937
1,Baseline,camera,1,0,216,0.004630,0.000000
2,Baseline,edge-infrastructure,1,0,72,0.013889,0.000000
3,Baseline,smart-plug,6,0,432,0.013889,0.000000
4,Baseline,network-device,1,0,72,0.013889,0.000000
5,Drift Simulation,network-device,432,430,432,1.000000,0.995370
6,Drift Simulation,edge-infrastructure,216,215,216,1.000000,0.995370
7,Drift Simulation,camera,3,2,648,0.004630,0.003086
8,Drift Simulation,smart-plug,38,5,1512,0.025132,0.003307
9,Drift Simulation,sensor,8,2,1512,0.005291,0.001323


,model,attack_family,attack_type,raw_detected_windows,confirmed_windows,total_windows,detected_events,total_events,raw_detection_rate,detection_rate,event_detection_rate
0,Drift Simulation,ddos,syn-flood-port-80,54,48,54,6,6,1.000000,0.888889,1.00
1,Drift Simulation,ddos,push-ack-flood-port-80,54,48,54,6,6,1.000000,0.888889,1.00
2,Drift Simulation,ddos,rst-fin-flood-port-80,54,48,54,6,6,1.000000,0.888889,1.00
3,Drift Simulation,ddos,udp-flood-port-80,54,48,54,6,6,1.000000,0.888889,1.00
4,Drift Simulation,ddos,rst-fin-flood-port-1883,63,56,63,7,7,1.000000,0.888889,1.00
...,...,...,...,...,...,...,...,...,...,...,...
145,Adaptive Fisher-Weighted Z,recon,ping-sweep,8,6,163,3,20,0.049080,0.036810,0.15
146,Adaptive Fisher-Weighted Z,dos,slowloris-port-8000,5,5,9,1,1,0.555556,0.555556,1.00
147,Adaptive Fisher-Weighted Z,dos,slowloris-port-554,7,7,9,1,1,0.777778,0.777778,1.00
148,Adaptive Fisher-Weighted Z,mitm,impersonation,0,0,56,0,7,0.000000,0.000000,0.00


,model,average_ms,median_ms,maximum_ms
0,Baseline,0.210097,0.2018,0.5623
1,Drift Simulation,0.204434,0.1983,0.4561
2,Adaptive Fisher-Weighted Z,0.286686,0.2784,0.6851


,storage_scope,items,bytes_per_item,total_bytes,details
0,general profile statistical state,5,608,3040,25 means + 25 standard deviations + 25 Fisher ...
1,client profile statistical state,20,608,12160,25 means + 25 standard deviations + 25 Fisher ...
2,stable rolling-statistics state,20,408,8160,count + 25 running means + 25 running M2 value...
3,peak drift rolling-statistics state,20,792,15840,rolling statistics + up to 48 scalar drift sco...
